In [ ]:
ON_COLAB  = False
import os
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"   # PLACEHOLDER: thesis inputs (read only)
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"   # PLACEHOLDER: new outputs
TEST_MODE = True
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [ ]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT, local_bin_dir=("/content" if ON_COLAB else None))
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

In [ ]:
from ddi.data import download_raw_data, load_data_into_dataframe
from ddi.paths import RAW_DATA_GDRIVE_ID

drug_db, event_db = download_raw_data(paths.raw_data_dir, RAW_DATA_GDRIVE_ID)
df = load_data_into_dataframe(drug_db, "drug")
lengths = df["smile"].str.len()
n = int((lengths > 256).sum())
print(f"{n} of {len(df)} drugs have SMILES > 256 characters "
      f"(longest {int(lengths.max())}, median {int(lengths.median())})")

In [ ]:
from ddi.data import get_all_events
ev = get_all_events(event_db)
ev = ev[ev["label"].astype(int) <= 227]
long_ids = set(df.loc[lengths > 256, "id"])
hit = ev["id1"].isin(long_ids) | ev["id2"].isin(long_ids)
print(f"{int(hit.sum()):,} of {len(ev):,} interactions ({100*hit.mean():.2f}%) involve a long-SMILES drug")